In [1]:
!pip install boto3 kaggle python-dotenv -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.8/14.8 MB 68.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 7.0 MB/s eta 0:00:00


In [6]:
import os
import glob
import boto3
import kagglehub
from botocore.exceptions import ClientError

In [ ]:
# ==========================================
# 1. SECRETS & AUTHENTICATION SETUP
# ==========================================
# Kaggle API Credentials
KAGGLE_API_TOKEN = os.environ.get("KAGGLE_API_TOKEN")

# AWS Credentials
AWS_ACCESS_KEY_ID = os.environ.get("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.environ.get("AWS_SECRET_ACCESS_KEY")

In [15]:
# ==========================================
# 2. ARCHITECTURE CONFIGURATION
# ==========================================
# Use the exact full bucket name you created in the AWS Console
BUCKET_NAME = "meli-recsys-mvp-iturriago-844339186350-us-east-2-an"
REGION = "us-east-2"
COMPETITION_HANDLE = "h-and-m-personalized-fashion-recommendations"

# MVP Constraints: Limit image upload to save time and S3 costs
MAX_IMAGES_TO_UPLOAD = 5000

# Initialize S3 Client
s3_client = boto3.client('s3', region_name=REGION)

def upload_file_to_s3(local_path: str, s3_key: str) -> bool:
    """
    Uploads a file to an S3 bucket with basic error handling.
    """
    try:
        s3_client.upload_file(local_path, BUCKET_NAME, s3_key)
        return True
    except ClientError as e:
        print(f"[ERROR] Failed to upload {local_path} to s3://{BUCKET_NAME}/{s3_key}: {e}")
        return False

In [10]:
# ==========================================
# 3. DATA ACQUISITION (KAGGLEHUB)
# ==========================================
print(f"[INFO] Downloading competition dataset '{COMPETITION_HANDLE}' via kagglehub...")
# kagglehub downloads and automatically extracts the contents to a local cache directory
dataset_path = kagglehub.competition_download(COMPETITION_HANDLE)
print(f"[INFO] Dataset downloaded and extracted successfully at: {dataset_path}")

[INFO] Downloading competition dataset 'h-and-m-personalized-fashion-recommendations' via kagglehub...


100%|██████████| 28.7G/28.7G [05:18<00:00, 96.6MB/s]

Extracting files...


[INFO] Dataset downloaded and extracted successfully at: /root/.cache/kagglehub/competitions/h-and-m-personalized-fashion-recommendations


In [16]:
# ==========================================
# 4. DATA LAKE INGESTION (S3 BRONZE LAYER)
# ==========================================

# 4.1 Upload Metadata and Interactions (CSVs)
# We upload the full CSVs because tabular data is lightweight and needed for the vector hybrid search
csv_files_to_upload = {
    "articles.csv": "bronze/metadata/articles.csv",
    "customers.csv": "bronze/metadata/customers.csv",
    "transactions_train.csv": "bronze/interactions/transactions_train.csv"
}

print("\n[INFO] Starting CSV uploads to S3 Bronze layer...")
for filename, s3_destination in csv_files_to_upload.items():
    local_file_path = os.path.join(dataset_path, filename)
    
    if os.path.exists(local_file_path):
        print(f"   -> Uploading {filename}...")
        upload_file_to_s3(local_file_path, s3_destination)
    else:
        print(f"[WARNING] File not found in downloaded dataset: {local_file_path}")

# 4.2 Upload a Subset of Images (Computer Vision inputs)
print(f"\n[INFO] Starting Image uploads (Capped at {MAX_IMAGES_TO_UPLOAD} for MVP)...")

# Glob pattern to find all .jpg files within the subdirectories of the images folder
images_pattern = os.path.join(dataset_path, "images", "**", "*.jpg")
all_local_images = glob.glob(images_pattern, recursive=True)

if not all_local_images:
    print("[WARNING] No images found in the downloaded dataset.")
else:
    images_to_process = all_local_images[:MAX_IMAGES_TO_UPLOAD]
    total_subset = len(images_to_process)
    
    print(f"   -> Found {len(all_local_images)} total images. Uploading {total_subset}...")
    
    successful_uploads = 0
    for i, img_local_path in enumerate(images_to_process):
        # Extract the relative path to maintain folder structure (e.g., 010/0108775015.jpg)
        # We split by 'images/' and take the second part
        relative_img_path = img_local_path.split(f"images{os.sep}")[-1]
        
        # Construct the S3 key: bronze/images/010/0108775015.jpg
        # Force forward slashes for S3 keys, even if running on Windows
        s3_image_key = f"bronze/images/{relative_img_path}".replace('\\', '/')
        
        if upload_file_to_s3(img_local_path, s3_image_key):
            successful_uploads += 1
            
        # Logging progress to avoid blind waits
        if (i + 1) % 500 == 0 or (i + 1) == total_subset:
            print(f"      Progress: {i + 1}/{total_subset} images uploaded to S3.")

print(f"\n[SUCCESS] Ingestion completed. {successful_uploads} images and core CSVs are now in your S3 Data Lake.")


[INFO] Starting CSV uploads to S3 Bronze layer...
   -> Uploading articles.csv...
   -> Uploading customers.csv...
   -> Uploading transactions_train.csv...

[INFO] Starting Image uploads (Capped at 5000 for MVP)...
   -> Found 105100 total images. Uploading 5000...
      Progress: 500/5000 images uploaded to S3.
      Progress: 1000/5000 images uploaded to S3.
      Progress: 1500/5000 images uploaded to S3.
      Progress: 2000/5000 images uploaded to S3.
      Progress: 2500/5000 images uploaded to S3.
      Progress: 3000/5000 images uploaded to S3.
      Progress: 3500/5000 images uploaded to S3.
      Progress: 4000/5000 images uploaded to S3.
      Progress: 4500/5000 images uploaded to S3.
      Progress: 5000/5000 images uploaded to S3.

[SUCCESS] Ingestion completed. 5000 images and core CSVs are now in your S3 Data Lake.
